## 1. ReAct Pattern (Reasoning + Acting)

The ReAct pattern combines reasoning with action:
1. **Thought**: Agent reasons about the problem
2. **Action**: Agent takes a specific action
3. **Observation**: Agent observes the result
4. **Repeat**: Loop until goal achieved

### Structure:
```
Thought: I need to find X
Action: search_database(query)
Observation: Result found
Thought: Now I need to analyze
Action: analyze_result(data)
Observation: Analysis complete
Final Answer: ...
```

In [ ]:
from dataclasses import dataclass
from typing import List, Dict, Any
from enum import Enum

class ActionType(Enum):
    SEARCH = "search"
    CALCULATE = "calculate"
    RETRIEVE = "retrieve"
    ANALYZE = "analyze"
    THINK = "think"
    FINAL_ANSWER = "final_answer"

@dataclass
class Thought:
    """Represents agent's reasoning"""
    content: str
    reasoning_type: str  # deductive, inductive, abductive

@dataclass
class Action:
    """Represents agent's action"""
    type: ActionType
    tool: str
    input_data: Dict[str, Any]

@dataclass
class Observation:
    """Represents environment feedback"""
    result: str
    success: bool
    metadata: Dict[str, Any]

@dataclass
class ReActStep:
    """Single step in ReAct loop"""
    thought: Thought
    action: Action
    observation: Observation

class ReActAgent:
    """Implements ReAct pattern for agentic reasoning"""
    
    def __init__(self, max_iterations: int = 10, verbose: bool = True):
        self.max_iterations = max_iterations
        self.verbose = verbose
        self.steps: List[ReActStep] = []
        self.tools = self._init_tools()
    
    def _init_tools(self) -> Dict[str, callable]:
        """Initialize available tools"""
        return {
            'search': self._tool_search,
            'calculate': self._tool_calculate,
            'retrieve': self._tool_retrieve,
            'analyze': self._tool_analyze,
        }
    
    def _tool_search(self, query: str) -> str:
        """Search tool - simulated"""
        results = {
            'python': 'Python is a high-level programming language',
            'ml': 'Machine Learning is a subset of AI',
            'agent': 'An agent is an autonomous system'
        }
        return results.get(query.lower(), 'No results found')
    
    def _tool_calculate(self, expression: str) -> str:
        """Calculate tool"""
        try:
            result = eval(expression)
            return f"Result: {result}"
        except:
            return "Calculation error"
    
    def _tool_retrieve(self, item_id: str) -> str:
        """Retrieve tool - simulated database"""
        data = {'001': 'Important data', '002': 'Critical info'}
        return data.get(item_id, 'Item not found')
    
    def _tool_analyze(self, data: str) -> str:
        """Analyze tool"""
        return f"Analysis: {data[:20]}... has been analyzed"
    
    def reason(self, problem: str) -> str:
        """Run ReAct loop"""
        if self.verbose:
            print(f"\n🤔 Problem: {problem}\n")
        
        current_context = problem
        
        for i in range(self.max_iterations):
            # Step 1: Think
            thought = self._generate_thought(current_context, i)
            if self.verbose:
                print(f"Thought {i+1}: {thought.content}")
            
            # Check if ready for final answer
            if "final answer" in thought.content.lower():
                return thought.content
            
            # Step 2: Act
            action = self._select_action(thought.content)
            if self.verbose:
                print(f"Action: {action.tool}({action.input_data})")
            
            # Step 3: Observe
            observation = self._execute_action(action)
            if self.verbose:
                print(f"Observation: {observation.result}\n")
            
            # Store step
            step = ReActStep(thought, action, observation)
            self.steps.append(step)
            
            # Update context
            current_context = observation.result
        
        return "Max iterations reached"
    
    def _generate_thought(self, context: str, iteration: int) -> Thought:
        """Simulate thought generation"""
        if iteration == 0:
            thought_content = f"Let me break down the problem: {context[:30]}..."
        else:
            thought_content = f"Based on previous findings, I should continue analyzing..."
        
        return Thought(
            content=thought_content,
            reasoning_type='deductive'
        )
    
    def _select_action(self, thought: str) -> Action:
        """Select appropriate action based on thought"""
        if 'search' in thought.lower():
            return Action(
                type=ActionType.SEARCH,
                tool='search',
                input_data={'query': 'agentic AI'}
            )
        elif 'calculate' in thought.lower():
            return Action(
                type=ActionType.CALCULATE,
                tool='calculate',
                input_data={'expression': '2 + 2'}
            )
        else:
            return Action(
                type=ActionType.RETRIEVE,
                tool='retrieve',
                input_data={'item_id': '001'}
            )
    
    def _execute_action(self, action: Action) -> Observation:
        """Execute action and return observation"""
        tool_func = self.tools.get(action.tool)
        
        if tool_func:
            if action.tool == 'calculate':
                result = tool_func(action.input_data['expression'])
            elif action.tool == 'search':
                result = tool_func(action.input_data['query'])
            elif action.tool == 'retrieve':
                result = tool_func(action.input_data['item_id'])
            else:
                result = tool_func(str(action.input_data))
            
            return Observation(
                result=result,
                success=True,
                metadata={'tool': action.tool, 'timestamp': 'now'}
            )
        
        return Observation(
            result='Tool not found',
            success=False,
            metadata={}
        )

# Example usage
agent = ReActAgent(verbose=True)
answer = agent.reason("What is the definition of agentic AI?")
print(f"\n✅ Final Answer: {answer}")

## 2. Chain-of-Thought (CoT) Pattern

CoT encourages explicit step-by-step reasoning before providing an answer:
- Breaks complex problems into smaller steps
- Shows intermediate reasoning
- Improves accuracy and transparency

### Variants:
- **Zero-shot CoT**: "Let's think step by step"
- **Few-shot CoT**: Provide examples with reasoning
- **Self-consistent CoT**: Sample multiple reasoning paths

In [ ]:
from typing import List, Tuple

class ReasoningStep:
    """Represents a single reasoning step"""
    def __init__(self, step_num: int, description: str, logic: str, conclusion: str):
        self.step_num = step_num
        self.description = description
        self.logic = logic
        self.conclusion = conclusion
    
    def __repr__(self):
        return f"Step {self.step_num}: {self.description}\nLogic: {self.logic}\nConclusion: {self.conclusion}"

class ChainOfThoughtAgent:
    """Implements Chain-of-Thought reasoning"""
    
    def __init__(self):
        self.reasoning_chain: List[ReasoningStep] = []
    
    def solve_problem(self, problem: str) -> Tuple[List[ReasoningStep], str]:
        """Solve using CoT"""
        self.reasoning_chain = []
        
        # Example: Solving a logic puzzle
        steps = self._generate_reasoning_steps(problem)
        
        for step in steps:
            self.reasoning_chain.append(step)
            print(f"\n{step}")
        
        final_answer = self._derive_final_answer()
        return self.reasoning_chain, final_answer
    
    def _generate_reasoning_steps(self, problem: str) -> List[ReasoningStep]:
        """Generate step-by-step reasoning"""
        steps = [
            ReasoningStep(
                1,
                "Understand the problem",
                f"Breaking down: {problem[:50]}...",
                "Problem identified"
            ),
            ReasoningStep(
                2,
                "Identify key components",
                "Extracting variables and constraints",
                "3 components identified"
            ),
            ReasoningStep(
                3,
                "Apply reasoning rules",
                "Using logical deduction",
                "Intermediate conclusion formed"
            ),
            ReasoningStep(
                4,
                "Verify logic",
                "Checking consistency",
                "Logic verified"
            )
        ]
        return steps
    
    def _derive_final_answer(self) -> str:
        """Derive final answer from reasoning chain"""
        if self.reasoning_chain:
            last_step = self.reasoning_chain[-1]
            return f"Therefore: {last_step.conclusion}"
        return "No conclusion"

# Example usage
cot_agent = ChainOfThoughtAgent()
chain, answer = cot_agent.solve_problem(
    "If all agents are systems, and all systems can learn, can all agents learn?"
)
print(f"\n✅ Final Answer: {answer}")

## 3. Tool-Using Agent Pattern

Agents use external tools to extend capabilities:
- **Tool Description**: Clear interface and usage
- **Tool Selection**: Choose appropriate tool
- **Tool Execution**: Run tool with inputs
- **Result Integration**: Use results in reasoning

In [ ]:
from abc import ABC, abstractmethod
from typing import Any, Dict
import json

class Tool(ABC):
    """Base class for agent tools"""
    
    @abstractmethod
    def name(self) -> str:
        pass
    
    @abstractmethod
    def description(self) -> str:
        pass
    
    @abstractmethod
    def input_schema(self) -> Dict[str, Any]:
        pass
    
    @abstractmethod
    def execute(self, **kwargs) -> str:
        pass

class WebSearchTool(Tool):
    def name(self) -> str:
        return "web_search"
    
    def description(self) -> str:
        return "Search the web for information"
    
    def input_schema(self) -> Dict[str, Any]:
        return {
            "query": {"type": "string", "description": "Search query"},
            "max_results": {"type": "integer", "default": 5}
        }
    
    def execute(self, query: str, max_results: int = 5) -> str:
        # Simulated search
        return f"Found {max_results} results for '{query}'"

class CalculatorTool(Tool):
    def name(self) -> str:
        return "calculator"
    
    def description(self) -> str:
        return "Perform mathematical calculations"
    
    def input_schema(self) -> Dict[str, Any]:
        return {
            "expression": {"type": "string", "description": "Mathematical expression"}
        }
    
    def execute(self, expression: str) -> str:
        try:
            result = eval(expression)
            return f"{expression} = {result}"
        except Exception as e:
            return f"Error: {str(e)}"

class DatabaseTool(Tool):
    def name(self) -> str:
        return "database"
    
    def description(self) -> str:
        return "Query structured data from database"
    
    def input_schema(self) -> Dict[str, Any]:
        return {
            "query_type": {"type": "string", "enum": ["select", "count", "filter"]},
            "table": {"type": "string"},
            "conditions": {"type": "object"}
        }
    
    def execute(self, query_type: str, table: str, conditions: dict = None) -> str:
        return f"Executed {query_type} query on {table}"

class ToolUsingAgent:
    """Agent that uses external tools"""
    
    def __init__(self):
        self.tools: Dict[str, Tool] = {}
        self.execution_history = []
        self._register_default_tools()
    
    def _register_default_tools(self):
        """Register default tools"""
        self.register_tool(WebSearchTool())
        self.register_tool(CalculatorTool())
        self.register_tool(DatabaseTool())
    
    def register_tool(self, tool: Tool):
        """Register a tool"""
        self.tools[tool.name()] = tool
    
    def get_available_tools(self) -> List[Dict]:
        """Get list of available tools"""
        tools_list = []
        for name, tool in self.tools.items():
            tools_list.append({
                "name": tool.name(),
                "description": tool.description(),
                "input_schema": tool.input_schema()
            })
        return tools_list
    
    def use_tool(self, tool_name: str, **kwargs) -> str:
        """Use a tool"""
        if tool_name not in self.tools:
            return f"Tool '{tool_name}' not found"
        
        tool = self.tools[tool_name]
        result = tool.execute(**kwargs)
        
        # Track execution
        self.execution_history.append({
            "tool": tool_name,
            "input": kwargs,
            "output": result
        })
        
        return result

# Example usage
agent = ToolUsingAgent()

print("Available Tools:")
for tool in agent.get_available_tools():
    print(f"\n✓ {tool['name']}: {tool['description']}")
    print(f"  Input: {tool['input_schema']}")

print("\n" + "="*50)
print("Tool Usage Examples:")
print("="*50)

# Use tools
result1 = agent.use_tool("web_search", query="agentic AI", max_results=10)
print(f"\nSearch Result: {result1}")

result2 = agent.use_tool("calculator", expression="2**10")
print(f"Calculator Result: {result2}")

result3 = agent.use_tool("database", query_type="select", table="agents")
print(f"Database Result: {result3}")

## 4. Hierarchical Agent Pattern

Multi-level agent structure for complex problem solving:
- **High-level Agent**: Strategic planning
- **Mid-level Agents**: Task decomposition
- **Low-level Agents**: Atomic actions

### Benefits:
- Modularity and reusability
- Scalability
- Easier debugging

In [ ]:
@dataclass
class AgentTask:
    """Represents a task for an agent"""
    id: str
    description: str
    priority: int
    status: str = "pending"
    subtasks: List['AgentTask'] = None
    
    def __post_init__(self):
        if self.subtasks is None:
            self.subtasks = []

class HierarchicalAgent:
    """Agent in hierarchical structure"""
    
    def __init__(self, name: str, level: int, capabilities: List[str]):
        self.name = name
        self.level = level  # 0=high, 1=mid, 2=low
        self.capabilities = capabilities
        self.assigned_tasks: List[AgentTask] = []
        self.completed_tasks: List[AgentTask] = []
    
    def can_handle_task(self, task: AgentTask) -> bool:
        """Check if agent can handle task"""
        # Simplified check
        return any(cap in task.description.lower() for cap in self.capabilities)
    
    def execute_task(self, task: AgentTask) -> bool:
        """Execute a task"""
        if self.can_handle_task(task):
            task.status = "completed"
            self.completed_tasks.append(task)
            return True
        return False

class HierarchicalAgentSystem:
    """Manages hierarchical agent structure"""
    
    def __init__(self):
        self.agents: Dict[int, List[HierarchicalAgent]] = {0: [], 1: [], 2: []}
        self.task_queue: List[AgentTask] = []
    
    def register_agent(self, agent: HierarchicalAgent):
        """Register agent at specific level"""
        self.agents[agent.level].append(agent)
    
    def submit_task(self, task: AgentTask):
        """Submit task to system"""
        self.task_queue.append(task)
    
    def dispatch_tasks(self):
        """Dispatch tasks to appropriate agents"""
        for task in self.task_queue:
            # Try to find appropriate agent
            assigned = False
            
            # Check each level from high to low
            for level in [0, 1, 2]:
                if assigned:
                    break
                
                for agent in self.agents[level]:
                    if agent.can_handle_task(task):
                        agent.assigned_tasks.append(task)
                        task.status = "assigned"
                        assigned = True
                        break
    
    def execute_all_tasks(self):
        """Execute all assigned tasks"""
        for level in [0, 1, 2]:
            for agent in self.agents[level]:
                for task in agent.assigned_tasks:
                    if task.status == "assigned":
                        agent.execute_task(task)
    
    def get_status(self):
        """Get system status"""
        status = {}
        for level, agents in self.agents.items():
            for agent in agents:
                status[agent.name] = {
                    "level": agent.level,
                    "assigned_tasks": len(agent.assigned_tasks),
                    "completed_tasks": len(agent.completed_tasks),
                    "capabilities": agent.capabilities
                }
        return status

# Example usage
print("🏗️ Hierarchical Agent System")
print("="*50)

system = HierarchicalAgentSystem()

# Create agents at different levels
planner = HierarchicalAgent("Planner", 0, ["plan", "strategy"])
coordinator1 = HierarchicalAgent("Coordinator-1", 1, ["search", "analyze"])
coordinator2 = HierarchicalAgent("Coordinator-2", 1, ["calculate", "retrieve"])
executor1 = HierarchicalAgent("Executor-1", 2, ["action", "execute"])

# Register agents
for agent in [planner, coordinator1, coordinator2, executor1]:
    system.register_agent(agent)

# Create and submit tasks
tasks = [
    AgentTask("t1", "Search for information", 1),
    AgentTask("t2", "Analyze results", 2),
    AgentTask("t3", "Calculate metrics", 1),
]

for task in tasks:
    system.submit_task(task)

# Dispatch and execute
system.dispatch_tasks()
system.execute_all_tasks()

# Print status
print("\nSystem Status:")
for agent_name, status in system.get_status().items():
    print(f"\n{agent_name}:")
    print(f"  Level: {status['level']}")
    print(f"  Assigned: {status['assigned_tasks']}, Completed: {status['completed_tasks']}")
    print(f"  Capabilities: {', '.join(status['capabilities'])}")

## 5. Feedback Loop Pattern

Agents learn from their actions and adjust behavior:
- **Action Execution**: Perform action
- **Feedback Collection**: Get results
- **Learning**: Update internal state
- **Adjustment**: Modify future behavior

In [ ]:
from collections import deque

@dataclass
class Experience:
    """Experience from agent action"""
    state: str
    action: str
    outcome: str
    reward: float
    timestamp: str

class LearningAgent:
    """Agent that learns from feedback"""
    
    def __init__(self, memory_size: int = 100):
        self.memory: deque = deque(maxlen=memory_size)
        self.action_success_rate: Dict[str, float] = {}
        self.preferred_actions: List[str] = []
    
    def take_action(self, action: str, state: str) -> Tuple[str, float]:
        """Take action and receive feedback"""
        # Simulate action with random outcome
        import random
        success = random.random() > 0.3  # 70% success rate
        reward = 1.0 if success else -0.5
        outcome = "success" if success else "failure"
        
        return outcome, reward
    
    def learn_from_experience(self, experience: Experience):
        """Learn from experience"""
        self.memory.append(experience)
        
        # Update action success rate
        if experience.action not in self.action_success_rate:
            self.action_success_rate[experience.action] = 0.0
        
        # Update success rate
        current_rate = self.action_success_rate[experience.action]
        is_success = 1.0 if experience.outcome == "success" else 0.0
        new_rate = 0.9 * current_rate + 0.1 * is_success
        self.action_success_rate[experience.action] = new_rate
    
    def get_best_actions(self, top_n: int = 3) -> List[str]:
        """Get best performing actions"""
        sorted_actions = sorted(
            self.action_success_rate.items(),
            key=lambda x: x[1],
            reverse=True
        )
        return [action for action, _ in sorted_actions[:top_n]]
    
    def select_action(self, available_actions: List[str]) -> str:
        """Select action based on learned preferences"""
        best_actions = self.get_best_actions()
        
        # Prefer best actions, with some randomness
        import random
        if random.random() < 0.7 and best_actions:
            return best_actions[0]
        else:
            return random.choice(available_actions)

# Example usage
print("📚 Learning Agent with Feedback Loop")
print("="*50)

learner = LearningAgent()
available_actions = ["search", "analyze", "calculate", "retrieve"]

# Run multiple trials
print("\nTraining Phase (100 iterations):")
for i in range(100):
    action = learner.select_action(available_actions)
    outcome, reward = learner.take_action(action, "state_1")
    
    experience = Experience(
        state="state_1",
        action=action,
        outcome=outcome,
        reward=reward,
        timestamp="now"
    )
    
    learner.learn_from_experience(experience)

print("\n📊 Learned Action Success Rates:")
for action, rate in sorted(
    learner.action_success_rate.items(),
    key=lambda x: x[1],
    reverse=True
):
    print(f"  {action}: {rate:.2%}")

print(f"\n🎯 Best Actions: {learner.get_best_actions()}")
print(f"📝 Experiences Stored: {len(learner.memory)}")

## 6. Planning Pattern

Agent uses planning to achieve goals efficiently:
- **Goal Decomposition**: Break into subgoals
- **Plan Generation**: Create action sequence
- **Plan Execution**: Execute step by step
- **Plan Adaptation**: Adjust to changes

In [ ]:
@dataclass
class Goal:
    """Represents agent goal"""
    id: str
    description: str
    priority: int
    deadline: str = None
    completed: bool = False

@dataclass  
class Plan:
    """Represents action plan"""
    goal_id: str
    steps: List[str]
    estimated_time: float
    dependencies: List[str] = None
    
    def __post_init__(self):
        if self.dependencies is None:
            self.dependencies = []

class PlanningAgent:
    """Agent that plans and executes"""
    
    def __init__(self):
        self.goals: List[Goal] = []
        self.plans: Dict[str, Plan] = {}
        self.execution_log: List[Dict] = []
    
    def set_goal(self, goal: Goal):
        """Set a goal for the agent"""
        self.goals.append(goal)
        goal_index = len(self.goals) - 1
        print(f"\n📍 Goal {goal.id} set: {goal.description}")
    
    def create_plan(self, goal: Goal) -> Plan:
        """Create plan for goal"""
        # Decompose goal into steps
        steps = self._decompose_goal(goal.description)
        estimated_time = len(steps) * 0.5  # 30 min per step
        
        plan = Plan(
            goal_id=goal.id,
            steps=steps,
            estimated_time=estimated_time
        )
        
        self.plans[goal.id] = plan
        print(f"\n📋 Plan created for goal {goal.id}:")
        for i, step in enumerate(steps, 1):
            print(f"   {i}. {step}")
        
        return plan
    
    def _decompose_goal(self, goal_description: str) -> List[str]:
        """Decompose goal into actionable steps"""
        # Generic decomposition
        return [
            "Analyze goal requirements",
            "Gather necessary resources",
            "Execute main task",
            "Verify results",
            "Document findings"
        ]
    
    def execute_plan(self, goal_id: str) -> bool:
        """Execute plan for goal"""
        if goal_id not in self.plans:
            return False
        
        plan = self.plans[goal_id]
        print(f"\n⚙️  Executing plan for goal {goal_id}:")
        
        for step in plan.steps:
            print(f"  ✓ Executing: {step}")
            self.execution_log.append({
                "goal_id": goal_id,
                "step": step,
                "status": "completed"
            })
        
        # Mark goal as completed
        for goal in self.goals:
            if goal.id == goal_id:
                goal.completed = True
                print(f"\n✅ Goal {goal_id} completed!")
        
        return True

# Example usage
print("🎯 Planning Agent System")
print("="*50)

planner = PlanningAgent()

# Set goals
goal1 = Goal(
    id="g1",
    description="Build and deploy an agentic AI system",
    priority=1,
    deadline="2024-12-31"
)

goal2 = Goal(
    id="g2",
    description="Create comprehensive agent documentation",
    priority=2,
    deadline="2024-12-15"
)

planner.set_goal(goal1)
planner.set_goal(goal2)

# Create and execute plans
for goal in planner.goals:
    plan = planner.create_plan(goal)
    planner.execute_plan(goal.id)

print(f"\n📊 Execution Summary:")
print(f"  Total goals: {len(planner.goals)}")
print(f"  Completed: {sum(1 for g in planner.goals if g.completed)}")
print(f"  Total steps executed: {len(planner.execution_log)}")

## Summary: Agent Design Patterns

| Pattern | Best For | Complexity | Key Components |
|---------|----------|-----------|------------------|
| ReAct | Complex reasoning | Medium | Thought→Action→Observation |
| CoT | Explainability | Low | Step-by-step reasoning |
| Tool Use | Capability extension | Medium | Tool interface, selection, execution |
| Hierarchical | Large systems | High | Multi-level agents, task decomposition |
| Feedback Loops | Continuous improvement | Medium | Experience, learning, adjustment |
| Planning | Goal achievement | High | Goal decomposition, plan execution |

## Key Takeaways

✅ **ReAct** combines reasoning with explicit action selection  
✅ **Chain-of-Thought** improves reasoning transparency  
✅ **Tool Use** extends agent capabilities  
✅ **Hierarchical** systems scale well  
✅ **Feedback loops** enable continuous learning  
✅ **Planning** drives goal-oriented behavior

## Exercises

### Exercise 1: ReAct with Custom Tools
Extend the ReAct agent to include 3+ custom tools and complex reasoning loops.

### Exercise 2: CoT Problem Solver
Build a Chain-of-Thought solver for logic puzzles with verification of each step.

### Exercise 3: Multi-Tool Agent
Create an agent that intelligently selects and chains multiple tools to solve problems.

### Exercise 4: Hierarchical System
Design a 3-level hierarchical agent system for managing a project.

### Exercise 5: Learning Optimization
Implement a learning agent that optimizes its action selection over time.

### Exercise 6: Planning Engine
Build a planning agent that handles complex goals with dependencies.